# NLP Group 7 Project P2

## Imports

In [8]:
from datasets import load_dataset
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
import os
import json

## Load dataset

In [9]:
dataset = load_dataset("MathArena/final_answer_comps", split="train")
df = dataset.to_pandas()

Generating train split: 100%|██████████| 139/139 [00:00<00:00, 4758.20 examples/s]


## General info and null values

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   problem_idx   139 non-null    int64 
 1   answer        139 non-null    object
 2   problem_type  130 non-null    object
 3   problem       139 non-null    object
 4   competition   139 non-null    object
 5   source        9 non-null      object
dtypes: int64(1), object(5)
memory usage: 6.6+ KB


In [ ]:
print("Dataset Head:\n", df.head(), sep="", end="\n" + "="*50 + "\n")
print("Dataset Columns:\n", df.columns, sep="", end="\n" + "="*50 + "\n")
print("Dataset Shape:\n", df.shape, sep="", end="\n" + "="*50 + "\n")

Dataset Head:
   problem_idx answer                    problem_type  \
0            1     70                 [Number Theory]   
1            2    588                      [Geometry]   
2            3     16                 [Combinatorics]   
3            4    117                       [Algebra]   
4            5    279  [Combinatorics, Number Theory]   

                                             problem     competition source  
0  Find the sum of all integer bases $b>9$ for wh...  aime/aime_2025   None  
1  On $\triangle ABC$ points $A, D, E$, and $B$ l...  aime/aime_2025   None  
2  The 9 members of a baseball team went to an ic...  aime/aime_2025   None  
3  Find the number of ordered pairs $(x,y)$, wher...  aime/aime_2025   None  
4  There are $8!= 40320$ eight-digit positive int...  aime/aime_2025   None  
Dataset Columns:
Index(['problem_idx', 'answer', 'problem_type', 'problem', 'competition',
       'source'],
      dtype='object')
Dataset Shape:
(139, 6)


## Communication with the model

In [12]:
load_dotenv()

ENDPOINT = "https://nlp-pcaf.services.ai.azure.com/openai/v1/"
MODEL_NAME = "DeepSeek-V3-0324"
DEPLOYMENT_NAME = "DeepSeek-V3-0324"

api_key = os.getenv("API_KEY")

client = OpenAI(
    base_url=f"{ENDPOINT}",
    api_key=api_key
)

completion = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?",
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionMessage(content='The capital of France is **Paris**. It is one of the most famous and visited cities in the world, known for landmarks like the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral.  \n\nWould you like any additional information about Paris or France? 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


In [13]:
def get_llm_response(role_prompt, user_input, temperature=0.0):
    """Sends a request to the Azure LLM with a System Role Prompt."""
    messages = [
        {"role": "system", "content": role_prompt},
        {"role": "user", "content": user_input}
    ]
    
    # Configure the response format for JSON
    response_format = {"type": "text"}
    if "JSON" in role_prompt or "JSON" in user_input:
         response_format = {"type": "json_object"}
    
    try:
        response = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=messages,
            temperature=temperature,
            response_format=response_format
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"LLM API Error: {e}"

## Role Prompts

In [18]:
VERIFIER_JSON_SCHEMA = """
{
  "valid": <boolean: true if final answer is correct, false otherwise>,
  "error_category": <string: one of 'CALCULATION_ERROR', 'CONCEPTUAL_FLAW', 'LOGIC_OMISSION', 'NONE'>,
  "critique_summary": <string: a brief, actionable explanation of the error>
}
"""

#SOLVER_ROLE = (
#    "You are the SOLVER, an expert mathematician. Your goal is to generate a detailed, "
#    "step-by-step Chain-of-Thought solution to the problem. Start with the initial setup. "
#    "If given a CRITIQUE, strictly follow it to correct your next attempt."
#)

VERIFIER_ROLE = (
    "You are the VERIFIER, a hyper-critical logic machine. Your sole task is to carefully analyze the "
    "provided solution and output a JSON object strictly adhering to the specified schema. "
    f"Analyze the solution for correctness and logic. Output strictly valid JSON only. No other comments, just a valid JSON object matching this schema: {VERIFIER_JSON_SCHEMA}"
    "Error categories are: 'CALCULATION_ERROR', 'CONCEPTUAL_FLAW', 'LOGIC_OMISSION', 'NONE'. "
)

# Example Problem
PROBLEM = (
    "A rectangular garden has sides in the ratio 4:3. If the area of the garden is 300 m^2, "
    "what is the length of the fence needed to enclose it? Provide your final answer as an integer."
)

# Example of a Flawed Solution (Solver's first attempt for PoC)
# This simulates the Solver making a calculation error: 2*(20+15) = 70, but the Solver will output 90.
FLAWED_SOLUTION_S1 = """
Solution:
1. Let the length be 4x and the width be 3x.
2. Area: (4x)(3x) = 12x^2.
3. 12x^2 = 300, so x^2 = 25, and x = 5.
4. Sides are 4(5)=20m and 3(5)=15m.
5. Perimeter P = 2(20 + 15) = 2(45) = 90m.
Final Answer: 90
"""

In [19]:
def get_solver_prompt(few_shot_examples=None):
    """
    Returns the System Prompt for the Solver.
    If few_shot_examples are provided, they are injected into the prompt.
    """
    base_prompt = (
    "You are the SOLVER, an expert mathematician. Your goal is to generate a detailed, "
    "step-by-step Chain-of-Thought solution to the problem. Start with the initial setup. "
)
    
    if few_shot_examples:
        base_prompt += f"\nHere are some examples of how to reason:\n{few_shot_examples}\n"
        
    base_prompt += "\nIf given a CRITIQUE, strictly follow it to correct your next attempt."
    return base_prompt

# FOR BASELINE (Current State):
SOLVER_ROLE = get_solver_prompt(few_shot_examples=None)

# FOR FUTURE PCAF (Future State):
# few_shots = "... content from MathInstruct ..."
# SOLVER_ROLE_PCAF = get_solver_prompt(few_shot_examples=few_shots)

In [20]:
import re

def extract_answer(text):
    """
    Extracts the final answer from LLM output.
    Priority 1: Look for \boxed{...} (Standard math format)
    Priority 2: Look for 'Final Answer: X' pattern
    Priority 3: Fallback to the very last number found in the text.
    """
    if not isinstance(text, str):
        return None

    # 1. Check for \boxed{answer} (Most reliable, used in your PoC)
    boxed_match = re.search(r'\\boxed\{([^}]+)\}', text)
    if boxed_match:
        return boxed_match.group(1).strip()

    # 2. Check for "Final Answer: <number>" pattern
    final_answer_match = re.search(r'(?:Final Answer|answer is)[:\s]*([-\d\.]+)', text, re.IGNORECASE)
    if final_answer_match:
        return final_answer_match.group(1).strip()

    # 3. Fallback: Find all numbers and return the last one
    # This handles cases where the model just ends with the number
    numbers = re.findall(r'[-+]?\d*\.\d+|\d+', text)
    if numbers:
        return numbers[-1]
        
    return None


In [21]:
def check_correctness(prediction, ground_truth):
    """
    Compares the extracted prediction with the ground truth.
    Handles type mismatches (string vs int) and float formatting.
    """
    if prediction is None or ground_truth is None:
        return False
        
    # Normalize to strings first
    pred_str = str(prediction).strip()
    gt_str = str(ground_truth).strip()
    
    # 1. Direct String Match
    if pred_str == gt_str:
        return True
        
    # 2. Numerical Match (Handles 70.0 vs 70)
    try:
        pred_float = float(pred_str)
        gt_float = float(gt_str)
        # Use a small epsilon for float comparison
        return abs(pred_float - gt_float) < 1e-6
    except ValueError:
        pass
        
    return False

## Test Case
The following code block is used to test the functionality of the above two functions (extract_answers and check_correctness)

In [23]:
# Test on your current dataframe head
print("--- Evaluation Pipeline Sanity Check ---")

# Let's test against the first few rows of your loaded dataframe
for index, row in df.head().iterrows():
    ground_truth = row['answer']
    
    # Simulate a "Perfect" LLM response using the \boxed{} format you saw in the PoC
    simulated_llm_output = f"After calculating, the answer is \\boxed{{{ground_truth}}}"
    
    # Run extraction
    extracted = extract_answer(simulated_llm_output)
    
    # Run correctness check
    is_correct = check_correctness(extracted, ground_truth)
    
    print(f"Prob ID {row['problem_idx']}: GT={ground_truth} | Extracted={extracted} | Correct? {is_correct}")

# Test a tricky failure case
print("\n--- Edge Case Test ---")
tricky_gt = 70
tricky_output = "The calculation gives 69.999 which rounds to 70. Final Answer: 70."
ext = extract_answer(tricky_output)
print(f"GT={tricky_gt} | Output='...Final Answer: 70.' | Extracted={ext} | Correct? {check_correctness(ext, tricky_gt)}")

--- Evaluation Pipeline Sanity Check ---
Prob ID 1: GT=70 | Extracted=70 | Correct? True
Prob ID 2: GT=588 | Extracted=588 | Correct? True
Prob ID 3: GT=16 | Extracted=16 | Correct? True
Prob ID 4: GT=117 | Extracted=117 | Correct? True
Prob ID 5: GT=279 | Extracted=279 | Correct? True

--- Edge Case Test ---
GT=70 | Output='...Final Answer: 70.' | Extracted=70. | Correct? True


# CoT Baseline
Here we generate the Chain-of-Thought baseline for N samples.

In [ ]:
import pandas as pd
import time

def run_baseline_evaluation(dataframe, num_samples=5):
    """
    Runs the Zero-Shot CoT Baseline (Solver only) on a subset of the dataframe.
    """
    results = []
    
    # 1. Select the subset (first N rows)
    # Using .copy() to ensure we don't accidentally modify the original slice
    subset = dataframe.head(num_samples).copy()
    
    print(f"--- Starting Baseline Run on {num_samples} problems ---")
    
    for index, row in subset.iterrows():
        problem_id = row['problem_idx']
        problem_text = row['problem']
        ground_truth = row['answer']
        
        print(f"Processing Problem ID: {problem_id}...", end=" ")
        
        # 2. Get Solver Response (Zero-Shot CoT)
        # Using the function and role you defined in main.ipynb
        try:
            llm_output = get_llm_response(SOLVER_ROLE, problem_text, temperature=0.0)
        except Exception as e:
            llm_output = f"ERROR: {str(e)}"
            print("API Fail")
        
        # 3. Extract and Score
        extracted_val = extract_answer(llm_output)
        is_correct = check_correctness(extracted_val, ground_truth)
        
        # Log to console for real-time tracking
        status = "PASS" if is_correct else "FAIL"
        print(f"{status} | GT: {ground_truth} | Pred: {extracted_val}")
        
        # 4. Record Data
        results.append({
            "problem_idx": problem_id,
            "problem_type": row['problem_type'],
            "ground_truth": ground_truth,
            "extracted_answer": extracted_val,
            "is_correct": is_correct,
            "llm_output": llm_output  # Keep full text for error analysis later
        })
        
        # Optional: Sleep briefly to avoid hitting tight rate limits if needed
        # time.sleep(0.5) 

    # 5. Compile Results
    results_df = pd.DataFrame(results)
    
    # Calculate Accuracy
    accuracy = results_df['is_correct'].mean() * 100
    print(f"\n--- Baseline Complete ---")
    print(f"Accuracy: {accuracy:.2f}% ({results_df['is_correct'].sum()}/{num_samples})")
    
    return results_df

# --- EXECUTION ---
# Run the baseline on 5 samples
baseline_results = run_baseline_evaluation(df, num_samples=30)

# Save to CSV for your records (as per Project Timeline Week 1)
baseline_results.to_csv("baseline_results_cutoff.csv", index=False)
print("\nResults saved to 'baseline_results_cutoff.csv'")

# Display the failure cases (to understand what the Verifier needs to catch)
print("\n--- Failure Analysis (Incorrect Rows) ---")
failures = baseline_results[~baseline_results['is_correct']]
if not failures.empty:
    print(failures[['problem_idx', 'ground_truth', 'extracted_answer']])
else:
    print("No failures in this small batch!")

--- Starting Baseline Run on 5 problems ---
Processing Problem ID: 1... PASS | GT: 70 | Pred: 70
Processing Problem ID: 2... PASS | GT: 588 | Pred: 588
Processing Problem ID: 3... PASS | GT: 16 | Pred: 16
Processing Problem ID: 4... PASS | GT: 117 | Pred: 117
Processing Problem ID: 5... PASS | GT: 279 | Pred: 279

--- Baseline Complete ---
Accuracy: 100.00% (5/5)

Results saved to 'baseline_results_cutoff.csv'

--- Failure Analysis (Incorrect Rows) ---
No failures in this small batch!


## PoC run

In [ ]:
# MatchArena's AIME first problem
dummy_problem = df.iloc[0][3]
solver_output = get_llm_response(SOLVER_ROLE, dummy_problem, temperature=0.0)
print(solver_output)

C:\Users\Felhasználó\AppData\Local\Temp\ipykernel_25856\3774873466.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dummy_problem = df.iloc[0][3]


### Understanding the Problem

First, I need to understand what the problem is asking. It's about finding all integer bases \( b > 9 \) such that the number \( 17_b \) (which is a number in base \( b \)) divides the number \( 97_b \) (another number in base \( b \)). After finding all such bases, I need to sum them up.

### Converting Base-b Numbers to Base-10

Since I'm more comfortable working in base-10, I think the first step is to convert \( 17_b \) and \( 97_b \) into base-10 expressions.

1. **Convert \( 17_b \) to base-10**:
   \[ 17_b = 1 \times b^1 + 7 \times b^0 = b + 7 \]

2. **Convert \( 97_b \) to base-10**:
   \[ 97_b = 9 \times b^1 + 7 \times b^0 = 9b + 7 \]

### Setting Up the Divisibility Condition

The problem states that \( 17_b \) must divide \( 97_b \). In base-10 terms, this means:
\[ (b + 7) \text{ divides } (9b + 7) \]

Mathematically, we can express this as:
\[ 9b + 7 \equiv 0 \mod (b + 7) \]

### Solving the Congruence

I recall that if \( a \equiv b \mod m \

In [ ]:
problem_solution_pairs = {
    dummy_problem: solver_output,
    PROBLEM: FLAWED_SOLUTION_S1
}

In [ ]:
for problem, solution in problem_solution_pairs.items():

    verifier_input = (
        f"Problem: {problem}\n\n"
        f"Solution to Critique:\n{solution}"
    )

    verifier_raw_output = get_llm_response(VERIFIER_ROLE, verifier_input, temperature=0.0)
    verifier_raw_output = verifier_raw_output.replace("```json", "")
    verifier_raw_output = verifier_raw_output.replace("```", "")
    verifier_raw_output = verifier_raw_output.strip()

    try:
        # JSON parsing
        verifier_json = json.loads(verifier_raw_output)
        print("JSON Parsing Successful. Verifier Output:")
        print(json.dumps(verifier_json, indent=2))
        
        # Check if the critique is valid and get correction signal
        if verifier_json.get('valid') == False:
            error_cat = verifier_json.get('error_category', 'UNKNOWN_ERROR')
            critique = verifier_json.get('critique_summary', 'No summary provided.')
            
            print("\nSolution is invalid. Preparing Planner's Correction...")
            
            # Planner logic
            planner_correction_hint = (
                f"CRITIQUE: The Verifier identified a **{error_cat}** at the final step. "
                f"Specifically: **{critique}**. You must rigorously re-examine your final calculation."
            )

            # Next Solver input
            solver_input_s2 = (
                f"ORIGINAL PROBLEM: {problem}\n\n"
                f"PREVIOUS FAILED ATTEMPT:\n{solution}\n\n"
                f"PLANNER'S CORRECTION HINT:\n{planner_correction_hint}\n\n"
                f"--- GENERATE CORRECTED SOLUTION (ATTEMPT S2) ---"
            )
            
        else:
            print("\nSolution passed. Loop terminates.")
            solver_input_s2 = None
            
    except json.JSONDecodeError as e:
        print(f"JSON Failure: Error: {e}")
        solver_input_s2 = None # Fail the loop

    # PCAF iteration 2 (Solver Correction)
    if solver_input_s2:
        print("\n--- 2. SOLVER CORRECTION (Targeted Generation Proof) ---")
        
        # Send the corrected prompt to the single LLM instance
        solver_corrected_output = get_llm_response(SOLVER_ROLE, solver_input_s2, temperature=0.0)
        
        print("Solver's Corrected Solution (S2):")
        print(solver_corrected_output)
        
        # We can run the verifier again and have multiple iterations but this is for PoC.
        print("\n[PoC Complete] The system successfully ran one full corrective loop.")
        print("The final correctness of S2 would be checked against the MathArena gold standard.")

JSON Parsing Successful. Verifier Output:
{
  "valid": true,
  "error_category": "NONE",
  "critique_summary": "The solution is correct and well-reasoned, with all steps logically sound and calculations accurate."
}

Solution passed. Loop terminates.
JSON Parsing Successful. Verifier Output:
{
  "valid": true,
  "error_category": "NONE",
  "critique_summary": "The solution correctly follows the perimeter calculation steps, with accurate calculations and logical flow."
}

Solution passed. Loop terminates.
